# Teacher inference (Pipeline **B v3.1**, **LongCLIP-B** 16+1 *scale-adaptive* view) — image + text → fused teacher → UMAP/HDBSCAN

**Why this revision (v3.1 vs v3).** B v3 brought back Pipeline A's 16+1 multiview, but used **fixed 16×16 native tiles** for every image. On the highest-resolution sources (~3000-px McCrone), 16 random 16×16 tiles cover ~0.1% of the image area — far below the global view's effective receptive field — so most of the fine-detail signal that motivated multiview in the first place was being thrown away. On the lowest-resolution sources (~150-px Emrath), 16×16 tiles cover ~1% of the image, the right ballpark. **B v3.1 closes that gap by sizing each tile per-image so it always covers a roughly constant fraction of the image's physical field of view.**

**Why B was changed (v3 vs v2 background).** B v2 (OpenAI ViT-L/14@336, global-only, 3072-D fused teacher) under-performed Pipeline A on the same data:

1. **Lost the long-context text tower.** OpenAI CLIP truncates every text column at 77 tokens, which silently drops the back half of long `Description` / `Extended description` / `composition` strings. LongCLIP-B keeps a **248-token** context, so the back half survives and contributes to the fused teacher.
2. **Lost the multi-scale image pool.** B v2's global-only image vector was a single short-side-224→336 forward. Pipeline A averaged a global view with **16 random native-resolution tiles**, recovering the high-frequency detail the global view loses on high-resolution McCrone / alaskafurid / Cameo-FRIL images.

**Pipeline B v3.1 = "Pipeline A's recipe under B's filenames + FOV-aware tiles"**: 16 native random *scale-adaptive* tiles + 1 global 224×224 view, on **LongCLIP-B** (ViT-B/16 visual + 248-token text tower). 512-D image branch + 3 × 512-D text → **2048-D fused teacher**.

**Pipeline B v3.1 view layout (per row):**

- **1 global view**: drop a 1-pixel border (`pre_crop_margin_px=1`) → short-side resize to **224** → center-crop **224×224**. The 1-pixel trim removes the typical scan-border artifact present on Pigment Compendium book scans.
- **16 random scale-adaptive native tiles**: drawn at the image's original resolution (no upsampling pre-extraction). The per-image tile side is computed as

  `tile_px = clip(round(B_TILE_MAX_PX × sqrt(W·H) / B_TILE_SCALE_REF_PX), B_TILE_MIN_PX, B_TILE_MAX_PX)`

  with defaults `B_TILE_MIN_PX = 16`, `B_TILE_MAX_PX = 224`, `B_TILE_SCALE_REF_PX = 3000`. So McCrone-class images (~3000-px geo-mean side) get full **224×224** tiles (the entire LongCLIP-B input — equivalent to drawing a fresh global view at a different position); Pigment Compendium / Emrath images saturate at **16×16** tiles (one ViT-B/16 patch token); Cameo-FRIL ~500-px images fall around **40×40**. CLIP's `preprocess` then up-samples every tile to 224×224 internally before the ViT sees it, so the network always sees a fixed-resolution input — the variation is purely in **what fraction of the FOV that input is computed from**. Set `B_TILE_FIXED_PX > 0` in the config cell to revert to the legacy fixed-size tiles (B v3).
- Crops are kept as **linear RGB in [0, 1]** with **no per-channel normalization** (the NEHM framework default in `nehm_pipeline/device.py` — preserves microscopy-relevant per-channel intensity ratios; the standard CLIP `(x − mean) / std` step would re-center channels and erase crossed-polars colors / Chelsea-filter behavior). This is enforced by stripping `Normalize` from LongCLIP's preprocess via `clip_visual_preprocess_without_channel_normalize(preprocess)`.
- **Image teacher** = weighted average `(IMAGE_WEIGHT_RANDOM·L2(random_pool) + IMAGE_WEIGHT_GLOBAL·L2(global)) / Σw`, then L2-normalized again. Default weights `0.5 / 0.5` (Pipeline A exact).

**Output dims:**

- `image_embeddings_B_random_pooled.npy` — `(N, 512)` — L2-normalized mean of 16 native tile embeddings.
- `image_embeddings_B_global.npy`        — `(N, 512)` — L2-normalized global-view embedding.
- per-column text npys                    — `(N, 512)` — LongCLIP text tower, 248-token context.
- **Fused teacher Z**                     — `(N, 2048)` = `[image_512 ⨁ illum_512 ⨁ avg(class+subclass)_512 ⨁ avg(desc+meta)_512]`.

**Output folders** (B v3.1 reuses the same `output_random_local_global_B/` directory; **delete its contents** before running because the previous B v2 OpenAI ViT-L 768/3072-D npys and the B v3 fixed-tile 512-D npys are not interchangeable with the new B v3.1 scale-adaptive 512-D npys without recomputing):

- `LongCLIP_Embeddings_v1/output_random_local_global_B/image_embeddings/`     (B v3 random + global npys + manifest)
- `LongCLIP_Embeddings_v1/output_random_local_global_B/column_text_embeddings/` (B v3 LongCLIP text npys + manifest)

**To clear old artifacts before this regen:**

```bash
rm -rf LongCLIP_Embeddings_v1/output_random_local_global_B/image_embeddings/*.npy
rm -rf LongCLIP_Embeddings_v1/output_random_local_global_B/image_embeddings/*.csv
rm -rf LongCLIP_Embeddings_v1/output_random_local_global_B/column_text_embeddings/*
```

**Cost.** Image regen is **17 LongCLIP-B forwards per row** (16 tiles + 1 global) — matches Pipeline A. Text regen is one LongCLIP-B text forward per (row × column) — minutes on A100, ~15-30 minutes on MPS. Both are resume-safe.

**Caveat.** A handful of Cameo-FRIL images (e.g. `2424.jpg`) have an asymmetric blackout region that's hundreds of pixels wide; a 1-pixel trim doesn't help, and tile sampling can land entirely inside the black region for those. If a few rows look obviously wrong in the image-only UMAP, that's the most likely cause.


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MATERIAL_DB_XLSX = PROJECT_ROOT / "Database" / "Material_Database.xlsx"

# B v3 reuses the same root as B v2; we will overwrite the npys/csvs here. Delete the contents
# of these directories before re-running this notebook (see the markdown cell at the top).
EMB_ROOT_B = PROJECT_ROOT / "LongCLIP_Embeddings_v1" / "output_random_local_global_B"
IMAGE_EMB_DIR_B = EMB_ROOT_B / "image_embeddings"
TEXT_EMB_DIR_B  = EMB_ROOT_B / "column_text_embeddings"
IMAGE_EMB_DIR_B.mkdir(parents=True, exist_ok=True)
TEXT_EMB_DIR_B.mkdir(parents=True, exist_ok=True)

IMAGE_EMB_RANDOM_B = IMAGE_EMB_DIR_B / "image_embeddings_B_random_pooled.npy"
IMAGE_EMB_GLOBAL_B = IMAGE_EMB_DIR_B / "image_embeddings_B_global.npy"
IMAGE_MANIFEST_B   = IMAGE_EMB_DIR_B / "image_manifest_B.csv"
TEXT_COLUMNS_MANIFEST_B = TEXT_EMB_DIR_B / "columns_manifest.csv"

IMAGES_DIR = PROJECT_ROOT / "NEHM"

# B v3 backbone — LongCLIP-B (ViT-B/16 visual, 248-token text).
LONGCLIP_ROOT = PROJECT_ROOT / "LongCLIP"
LONGCLIP_CHECKPOINT = LONGCLIP_ROOT / "checkpoints" / "longclip-B.pt"
EXPECTED_IMAGE_EMBED_DIM = 768   # ViT-B/16: pooled ln_post CLS (no visual.proj)
EXPECTED_TEXT_EMBED_DIM = 512   # EOT after ln_final (no text_projection)
EXPECTED_FUSED_DIM = EXPECTED_IMAGE_EMBED_DIM + 3 * EXPECTED_TEXT_EMBED_DIM  # 2304

# B v3.1 view layout — 17 views per row = 16 SCALE-ADAPTIVE random tiles + 1 global.
# Each tile's native side is computed per-image as
#   tile_px = clip(round(TILE_MAX_PX * sqrt(W*H) / TILE_SCALE_REF_PX), TILE_MIN_PX, TILE_MAX_PX)
# so the tile fraction-of-FOV is roughly constant across sources (the dataset spans
# ~150 px short-side at the smallest end up to ~3020 px on McCrone). Larger images get
# bigger tiles (saturating at 224 = LongCLIP-B input); smaller images get smaller tiles
# (saturating at 16 = the ViT-B/16 patch size).
B_GLOBAL_PX        = 224    # LongCLIP-B native input size
B_TILE_FIXED_PX    = 0      # 0 = use the scale-adaptive sizing below; >0 = legacy fixed-size tiles
B_TILE_MIN_PX      = 16     # smallest adaptive tile side (matches ViT-B/16 patch grid)
B_TILE_MAX_PX      = 224    # largest adaptive tile side (matches LongCLIP-B input)
B_TILE_SCALE_REF_PX = 3000  # sqrt(W*H) image side that maps to TILE_MAX_PX
B_N_TILES          = 16     # native random tiles per image (per Pipeline A)
PRE_CROP_MARGIN_PX = 1      # drop 1px scan-border before short-side resize

# Image-teacher weighted average (matches Pipeline A's recipe exactly).
IMAGE_WEIGHT_RANDOM = 0.5
IMAGE_WEIGHT_GLOBAL = 0.5

print(f"project root        : {PROJECT_ROOT}")
print(f"image dir           : {IMAGES_DIR}")
print(f"B v3.1 image emb dir: {IMAGE_EMB_DIR_B}")
print(f"B v3.1 text  emb dir: {TEXT_EMB_DIR_B}")
print(f"backbone (B v3.1)   : LongCLIP-B  (768-D image ln_post + 512-D text pre-proj, 248-tok)")
print(f"checkpoint          : {LONGCLIP_CHECKPOINT}  exists={LONGCLIP_CHECKPOINT.is_file()}")
if int(B_TILE_FIXED_PX) > 0:
    print(
        f"view layout         : 1 global @ {B_GLOBAL_PX}x{B_GLOBAL_PX} + "
        f"{B_N_TILES}× {B_TILE_FIXED_PX}x{B_TILE_FIXED_PX} fixed-size native tiles (legacy)"
    )
else:
    print(
        f"view layout         : 1 global @ {B_GLOBAL_PX}x{B_GLOBAL_PX} + "
        f"{B_N_TILES}× scale-adaptive native tiles "
        f"[{B_TILE_MIN_PX}-{B_TILE_MAX_PX}px @ ref={B_TILE_SCALE_REF_PX}]"
    )
print(f"image teacher weights: random={IMAGE_WEIGHT_RANDOM}  global={IMAGE_WEIGHT_GLOBAL}")
print(f"pre_crop_margin_px  : {PRE_CROP_MARGIN_PX}  (drop scan-border pixels before resize)")


In [ ]:
_df_raw = pd.read_excel(MATERIAL_DB_XLSX, sheet_name="Sheet2")
if "Image Index" not in _df_raw.columns:
    raise ValueError(f"Expected 'Image Index' in {MATERIAL_DB_XLSX} sheet 'Sheet2'")

_df_manifest = _df_raw.reset_index(drop=True).copy()
N = int(len(_df_manifest))
row_indices = np.arange(N, dtype=np.int64)

MANIFEST_KEYS = _df_manifest["Image Index"].astype(np.int64).to_numpy()
if "Image" in _df_manifest.columns:
    filenames = _df_manifest["Image"].fillna("").astype(str).to_numpy()
else:
    filenames = np.array(["" for _ in range(N)], dtype=object)

y_sheet_default = np.full(N, -1, dtype=np.int64)
print(f"Rows loaded: {N}")


## Load LongCLIP-B (ViT-B/16, 512-D, 248-token text)

The LongCLIP package is structured as a relative-import package (`from .model_longclip import
build_model`), so the `model/` directory must be loaded *as a package*, not as a loose file.
The loader below mirrors the pattern in `Notebooks/Student_inference_embeddings.ipynb` and
`Colab/LongCLIP_Student_Trainer.ipynb`: register `LongCLIP/` on `sys.path`, evict any stale
`model.*` entries from `sys.modules`, and import `model.longclip`.

After loading, we strip torchvision's `Normalize(mean, std)` from the returned `preprocess`
via `nehm_pipeline.device.clip_visual_preprocess_without_channel_normalize` so RGB stays
linear in `[0, 1]` (NEHM framework default — preserves microscopy-relevant per-channel
intensity ratios). The trainer and inference notebooks do the same thing; this is what keeps
teacher / student / inference all aligned on the same pixel scale.


In [ ]:
import importlib
import importlib.util

import torch

from nehm_pipeline.device import (
    clip_visual_preprocess_without_channel_normalize,
    resolve_device,
)


DEVICE = resolve_device("cuda" if torch.cuda.is_available() else "mps")
print(f"device for regen: {DEVICE}")


def _load_longclip_module():
    """Robust LongCLIP package loader (mirrors the pattern in trainer / inference A)."""
    pkg_dir = LONGCLIP_ROOT / "model"
    init_py = pkg_dir / "__init__.py"
    longclip_py = pkg_dir / "longclip.py"
    model_longclip_py = pkg_dir / "model_longclip.py"
    simple_tok_py = pkg_dir / "simple_tokenizer.py"
    missing = [p for p in (pkg_dir, init_py, longclip_py, model_longclip_py, simple_tok_py) if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "LongCLIP package incomplete; missing:\n  " + "\n  ".join(str(m) for m in missing)
        )
    if str(LONGCLIP_ROOT) not in sys.path:
        sys.path.insert(0, str(LONGCLIP_ROOT))
    for k in [k for k in list(sys.modules) if k == "model" or k.startswith("model.")]:
        del sys.modules[k]
    try:
        from model import longclip as _lc  # type: ignore
        return _lc
    except Exception as _err:
        print(f"  (simple `from model import longclip` failed: {_err}; falling back to importlib)")
    pkg_spec = importlib.util.spec_from_file_location(
        "model", str(init_py), submodule_search_locations=[str(pkg_dir)],
    )
    pkg_mod = importlib.util.module_from_spec(pkg_spec)
    sys.modules["model"] = pkg_mod
    pkg_spec.loader.exec_module(pkg_mod)
    sub_spec = importlib.util.spec_from_file_location("model.longclip", str(longclip_py))
    sub_mod = importlib.util.module_from_spec(sub_spec)
    sys.modules["model.longclip"] = sub_mod
    sub_spec.loader.exec_module(sub_mod)
    return sub_mod


if not LONGCLIP_CHECKPOINT.is_file():
    raise FileNotFoundError(f"LongCLIP checkpoint not found: {LONGCLIP_CHECKPOINT}")

longclip_module = _load_longclip_module()
print(f"LongCLIP loaded from: {LONGCLIP_ROOT / 'model' / 'longclip.py'}")

# longclip.load returns (model, preprocess) just like clip.load, with a torchvision Compose
# preprocess that includes Normalize(mean, std). We strip Normalize so RGB stays linear in [0, 1].
clip_model, _preprocess_with_normalize = longclip_module.load(str(LONGCLIP_CHECKPOINT), device=str(DEVICE))
clip_model = clip_model.float().to(DEVICE)
clip_model.eval()
preprocess = clip_visual_preprocess_without_channel_normalize(_preprocess_with_normalize)
print("LongCLIP-B preprocess: Normalize stripped (linear RGB in [0, 1])")

# Probe both heads — **pre-projection** teacher (768-D image ln_post, 512-D text).
from PIL import Image
from nehm_pipeline.longclip_image_embeddings import (
    vit_visual_ln_post_forward,
    clip_longclip_encode_text_after_ln_final,
)
_probe = Image.new("RGB", (B_GLOBAL_PX, B_GLOBAL_PX), (127, 127, 127))
with torch.inference_mode():
    _t = preprocess(_probe).unsqueeze(0).to(DEVICE)
    _e_img = vit_visual_ln_post_forward(clip_model.visual, _t)
print(f"image tower output_dim = {int(_e_img.shape[-1])} (expected {EXPECTED_IMAGE_EMBED_DIM})")

_tokenize = getattr(longclip_module, "tokenize", None)
if _tokenize is None:
    _tokenize = getattr(getattr(longclip_module, "simple_tokenizer", object()), "tokenize", None)
if _tokenize is None:
    raise AttributeError("Could not locate `tokenize` on the loaded LongCLIP module.")
try:
    _tok = _tokenize(["probe"], truncate=True).to(DEVICE)
except TypeError:
    _tok = _tokenize(["probe"]).to(DEVICE)
with torch.inference_mode():
    _e_txt = clip_longclip_encode_text_after_ln_final(clip_model, _tok)
print(f"text  tower output_dim = {int(_e_txt.shape[-1])} (expected {EXPECTED_TEXT_EMBED_DIM})")

if int(_e_img.shape[-1]) != EXPECTED_IMAGE_EMBED_DIM or int(_e_txt.shape[-1]) != EXPECTED_TEXT_EMBED_DIM:
    raise RuntimeError(
        f"Backbone dim mismatch. Expected image={EXPECTED_IMAGE_EMBED_DIM}, text={EXPECTED_TEXT_EMBED_DIM}; got "
        f"image={int(_e_img.shape[-1])}, text={int(_e_txt.shape[-1])}."
    )


## Step A — Regenerate **B v3.1 image embeddings** (minutes-to-tens-of-minutes; resume-safe)

For every row in `_df_manifest`, encodes **17 LongCLIP-B forwards per row**: 16 random
**scale-adaptive** native-resolution tiles (mean-pooled and re-L2'd) **plus** one global
view (drop a 1-pixel border → short-side resize to 224 → center-crop 224×224).

Each tile's native side is chosen per-image as

```
tile_px = clip(round(B_TILE_MAX_PX * sqrt(W*H) / B_TILE_SCALE_REF_PX),
               B_TILE_MIN_PX, B_TILE_MAX_PX)
```

with the constants you set in the config cell (defaults `B_TILE_MIN_PX = 16`,
`B_TILE_MAX_PX = 224`, `B_TILE_SCALE_REF_PX = 3000`). This means the largest images
(McCrone ~3000-px short side) get full-resolution 224×224 tiles, while the smallest
images (Emrath ~150-px) saturate at 16×16 tiles, and every tile in between samples a
roughly constant fraction of the *physical field of view*. Set `B_TILE_FIXED_PX > 0`
in the config cell to force every tile to the same legacy fixed size instead.

Writes:

- `image_embeddings_B_random_pooled.npy` — `(N, 512)` weighted-pooled tile embeddings
- `image_embeddings_B_global.npy`        — `(N, 512)` global-view embedding
- `image_manifest_B.csv`                 — per-row resolved path, `exists`,
  `image_w_px`, `image_h_px`, `image_geo_px`, **`tile_px_used`**

The new manifest columns let you sanity-check the adaptive scaling — Step A prints a
histogram of the tile sizes actually used (rounded to nearest 16 px) at the end.

On A100 the full 5,410-row regen takes ~2-3 minutes; on MPS expect ~15-20 minutes. The run
is resume-safe (set `FORCE_RECOMPUTE = False` to skip rows that already have finite
embeddings on disk). Smoke-test first by setting `MAX_ROWS = 50, FORCE_RECOMPUTE = True`.


In [ ]:
from nehm_pipeline.longclip_image_embeddings import (
    LongClipBImageEmbedConfig,
    regenerate_longclip_b_image_embeddings,
)

FORCE_RECOMPUTE = True   # set True to ignore existing npys and re-encode every row
MAX_ROWS = 0             # 0 = full DB; otherwise smoke-test the first MAX_ROWS rows

_n_rows = int(MAX_ROWS) if MAX_ROWS and MAX_ROWS > 0 else N
_filenames = list(filenames[:_n_rows])

if FORCE_RECOMPUTE:
    for _stale in (IMAGE_EMB_GLOBAL_B, IMAGE_EMB_RANDOM_B, IMAGE_MANIFEST_B):
        if _stale.is_file():
            _stale.unlink()
            print(f"  removed existing: {_stale.name}")

cfg_img_b = LongClipBImageEmbedConfig(
    n_random_crops=B_N_TILES,
    random_crop_px=B_TILE_FIXED_PX,        # 0 = scale-adaptive (B v3.1); >0 = fixed-size legacy
    tile_min_px=B_TILE_MIN_PX,
    tile_max_px=B_TILE_MAX_PX,
    tile_scale_ref_px=B_TILE_SCALE_REF_PX,
    global_target_px=B_GLOBAL_PX,
    pre_crop_margin_px=PRE_CROP_MARGIN_PX,
    image_batch_size=32,
    random_seed=1337,
)
print(f"B v3.1 image regen config: {cfg_img_b}")

_rand_path, _glob_path, _manifest_path = regenerate_longclip_b_image_embeddings(
    clip_model=clip_model,
    preprocess=preprocess,
    device=DEVICE,
    image_dir=IMAGES_DIR,
    image_filenames=_filenames,
    n_rows=_n_rows,
    embed_dim=EXPECTED_IMAGE_EMBED_DIM,
    out_dir=IMAGE_EMB_DIR_B,
    cfg=cfg_img_b,
    use_encode_image=True,
    progress=True,
    skip_existing_rows=not FORCE_RECOMPUTE,
)
print(f"B v3.1 random pool: {_rand_path}")
print(f"B v3.1 global     : {_glob_path}")
print(f"B image manifest  : {_manifest_path}")
assert _rand_path is not None, "B v3.1 must produce a random-pooled npy (n_random_crops > 0)"

# Quick sanity check on the manifest's per-row tile sizes (only meaningful in adaptive mode).
try:
    _man = pd.read_csv(_manifest_path)
    if "tile_px_used" in _man.columns:
        used = _man.loc[_man["tile_px_used"] > 0, "tile_px_used"].astype(int)
        if len(used):
            print(
                f"adaptive tile sizes: min={used.min()}  median={int(used.median())}  "
                f"mean={used.mean():.1f}  max={used.max()}  "
                f"(over {len(used)}/{len(_man)} successfully encoded rows)"
            )
            print(
                "  histogram (rounded to nearest 16):\n"
                + str((used // 16 * 16).value_counts().sort_index().to_dict())
            )
except Exception as _err:
    print(f"  (skipping tile-size summary: {_err})")


## Step B — Regenerate **B v3 text embeddings** with LongCLIP-B (248-token context)

For every column in the DB sheet (skipping `Image` and `Image Index`), encode each row's
string value through **LongCLIP-B's text tower** with its **248-token context window** (vs
CLIP's 77). This is the key piece B v2 was missing: long `Description`, `Extended description`,
and `composition` strings are no longer truncated at the comma.

Each column's `(N, 512)` matrix is saved to `column_text_embeddings/<col>.npy`; a
`columns_manifest.csv` indexes the lot in the same format the fusion cell expects.

If LongCLIP errors on a single row (very rarely it does for text containing unusual control
chars), the helper falls back to encoding without `truncate=True`. If the entire batch
errors, expect a clear traceback so you can locate the offending row.


In [ ]:
from nehm_pipeline.longclip_image_embeddings import regenerate_longclip_b_text_embeddings

# Encode all text columns of Sheet2 except 'Image' and 'Image Index' (covered by image regen).
# Smaller batch than CLIP because LongCLIP's 248-token sequences are ~3x more KV memory.
_text_manifest_path = regenerate_longclip_b_text_embeddings(
    clip_model=clip_model,
    longclip_module=longclip_module,
    device=DEVICE,
    df=_df_manifest,
    out_dir=TEXT_EMB_DIR_B,
    text_batch_size=64,
    embed_dim=EXPECTED_TEXT_EMBED_DIM,
    skip_columns=("Image", "Image Index"),
    progress=True,
)
print(f"B v3 text manifest: {_text_manifest_path}")
print("first few text columns:")
print(pd.read_csv(_text_manifest_path).head(8).to_string(index=False))


## Step C — Build the fused **B v3 teacher**  Z = (N, 2048)

`Z = [image_only_512 ⨁ illum_512 ⨁ avg(class+subclass)_512 ⨁ avg(desc+meta)_512] = (N, 2048)`.

In B v3, `image_only = L2((IMAGE_WEIGHT_RANDOM·L2(random_pool) + IMAGE_WEIGHT_GLOBAL·L2(global)) / Σw)`
— exactly Pipeline A's recipe. Default weights are `0.5 / 0.5`. The per-column text averages
reuse the A masking logic (skip rows whose source string is empty/`nan`, and weight
`composition` 2× inside `avg(desc+meta)`).


In [ ]:
from nehm_pipeline.longclip_image_embeddings import fill_missing_with_zeros

for pth in [TEXT_COLUMNS_MANIFEST_B, IMAGE_EMB_GLOBAL_B, IMAGE_EMB_RANDOM_B]:
    if not Path(pth).is_file():
        raise FileNotFoundError(f"Missing required file: {pth}")

_need_rows = int(np.max(row_indices)) + 1 if len(row_indices) else 0

img_glob_full = np.load(IMAGE_EMB_GLOBAL_B).astype(np.float32)
img_rand_full = np.load(IMAGE_EMB_RANDOM_B).astype(np.float32)
if img_glob_full.shape[0] < _need_rows or img_rand_full.shape[0] < _need_rows:
    raise ValueError("Image embedding array does not cover required row range")

img_glob = fill_missing_with_zeros(img_glob_full[row_indices])
img_rand = fill_missing_with_zeros(img_rand_full[row_indices])


def _l2_rows(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)


# B v3 image teacher = L2(weighted avg of L2(random_pool), L2(global))  (Pipeline A's recipe).
_w_sum = float(IMAGE_WEIGHT_RANDOM) + float(IMAGE_WEIGHT_GLOBAL)
if _w_sum <= 0:
    raise ValueError(
        f"IMAGE_WEIGHT_RANDOM + IMAGE_WEIGHT_GLOBAL must be > 0 "
        f"(got {IMAGE_WEIGHT_RANDOM} + {IMAGE_WEIGHT_GLOBAL})."
    )
_image_blend = (
    float(IMAGE_WEIGHT_RANDOM) * _l2_rows(img_rand)
    + float(IMAGE_WEIGHT_GLOBAL) * _l2_rows(img_glob)
) / _w_sum
image_only = _l2_rows(_image_blend).astype(np.float32)
if image_only.shape[1] != EXPECTED_IMAGE_EMBED_DIM:
    raise ValueError(
        f"image_only dim {image_only.shape[1]} != {EXPECTED_IMAGE_EMBED_DIM}; "
        f"check the teacher regen ran with LongCLIP-B (512-D)."
    )

manifest_df = pd.read_csv(TEXT_COLUMNS_MANIFEST_B)
if "column_name" not in manifest_df.columns or "embedding_file" not in manifest_df.columns:
    raise ValueError(f"Unexpected columns manifest format: {TEXT_COLUMNS_MANIFEST_B}")

col_emb = {}
for _, r in manifest_df.iterrows():
    col = str(r["column_name"])
    _raw_emb = Path(r["embedding_file"])
    emb_path = _raw_emb if (_raw_emb.is_absolute() and _raw_emb.is_file()) else (TEXT_EMB_DIR_B / _raw_emb.name)
    arr = np.load(emb_path).astype(np.float32)
    if arr.shape[0] < _need_rows:
        raise ValueError(f"Column emb rows for {col} ({arr.shape[0]}) do not cover required row range")
    if arr.shape[1] != EXPECTED_TEXT_EMBED_DIM:
        raise ValueError(
            f"Column emb dim for {col} ({arr.shape[1]}) != {EXPECTED_TEXT_EMBED_DIM}; "
            f"regenerate text embeddings under LongCLIP-B (Step B above)."
        )
    col_emb[col] = arr[row_indices]

for bad_col in ["Image Index", "Image"]:
    col_emb.pop(bad_col, None)

required_text = [
    "Illumination Modality", "Class", "Subclass", "Description", "Stain",
    "Magnification", "composition", "refractive index",
]
for c in required_text:
    if c not in col_emb:
        raise KeyError(f"Missing required text column embedding: {c}")

illum = col_emb["Illumination Modality"]
group_class_subclass = ["Class", "Subclass"]
group_desc_meta = ["Description", "Stain", "Magnification", "composition", "refractive index"]
composition_weight = 2.0


def _masked_group_avg(cols, weights=None):
    d = next(iter(col_emb.values())).shape[1]
    s = np.zeros((N, d), dtype=np.float32)
    cnt = np.zeros((N, 1), dtype=np.float32)
    for col in cols:
        w = 1.0 if weights is None else float(weights.get(col, 1.0))
        valid = (
            _df_manifest[col].fillna("").astype(str)
            .map(lambda x: x.strip() != "" and x.strip().lower() != "nan")
            .to_numpy(dtype=np.float32).reshape(N, 1)
        ) if col in _df_manifest.columns else np.zeros((N, 1), dtype=np.float32)
        vw = valid * w
        s += col_emb[col] * vw
        cnt += vw
    out = (s / np.maximum(cnt, 1.0)).astype(np.float32)
    out[cnt[:, 0] == 0.0] = 0.0
    return out, cnt


avg_class_subclass, _ = _masked_group_avg(group_class_subclass)
avg_desc_meta, _ = _masked_group_avg(group_desc_meta, weights={"composition": composition_weight})

Z = np.concatenate([image_only, illum, avg_class_subclass, avg_desc_meta], axis=1).astype(np.float32)
if Z.shape[1] != EXPECTED_FUSED_DIM:
    raise ValueError(
        f"Fused Z dim {Z.shape[1]} != {EXPECTED_FUSED_DIM}; check that image_only and all "
        f"required text columns are {EXPECTED_TEXT_EMBED_DIM}-D."
    )
y_sheet = y_sheet_default.copy()

print(
    f"Built Z (B v3 teacher) {Z.shape}  |  image_only = "
    f"L2({IMAGE_WEIGHT_RANDOM}·L2(random) + {IMAGE_WEIGHT_GLOBAL}·L2(global))"
)
print(
    f"Block layout : image_{EXPECTED_IMAGE_EMBED_DIM} + illum_{EXPECTED_TEXT_EMBED_DIM} + "
    f"avg(class+subclass)_{EXPECTED_TEXT_EMBED_DIM} + avg(desc+meta)_{EXPECTED_TEXT_EMBED_DIM} "
    f"= {EXPECTED_FUSED_DIM}-D"
)


## Step D — UMAP + HDBSCAN on the B v3 teacher

Same recipe as the v1 / A teacher notebook (5D UMAP → HDBSCAN → 2D UMAP for plotting).
Cluster IDs are saved with a `_B` suffix so they don't overwrite A's labels. The 2048-D
fused teacher (LongCLIP-B) tends to produce **fewer-and-cleaner** clusters and lower noise
than the 3072-D OpenAI ViT-L/14 teacher because (a) the long text columns now contribute,
and (b) the random-tile pool is back. Expect noise fraction in the 5-15% range, vs 25-30%
for B v2.


In [ ]:
import warnings

import umap
import hdbscan
from sklearn.preprocessing import normalize

UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.1
UMAP_METRIC = "cosine"
CLUSTER_MIN_SIZE = 12
CLUSTER_MIN_SAMPLES = 2
CLUSTER_SELECTION_METHOD = "eom"

Zn = normalize(Z, norm="l2", axis=1)

# UMAP 0.5.x emits a UserWarning every time you set random_state ("n_jobs value 1 overridden to 1
# ... Use no seed for parallelism."). Reproducibility matters more than parallelism here, so we
# silence just that message without touching other warnings.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r".*n_jobs value .* overridden to 1 by setting random_state.*",
        category=UserWarning,
        module=r"umap\..*",
    )

    umap5 = umap.UMAP(
        n_components= 15,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=42,
    )
    z5 = umap5.fit_transform(Zn)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=CLUSTER_MIN_SIZE,
        min_samples=CLUSTER_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method=CLUSTER_SELECTION_METHOD,
    )
    hdb_labels = clusterer.fit_predict(z5)

    umap2 = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=42,
    )
    xy = umap2.fit_transform(Zn)

n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
noise_frac = float((hdb_labels == -1).sum()) / float(N)
print(f"Clusters (B v3): {n_clusters} | noise fraction: {noise_frac:.3f} | points: {N}")


## Step E — Interactive UMAP plot (Plotly, hover-on)

2-D UMAP scatter colored by HDBSCAN cluster (B v3 teacher). Hover any point to see the row's
`Image Index`, filename, `Source`, `Class`, `Subclass`, and assigned cluster. Noise points
(`hdbscan_cluster_B == -1`) are drawn first in grey so they sit underneath the colored clusters.

The plot is rendered inline with the default Plotly notebook renderer; it also writes a
self-contained `.html` next to the cluster CSV so you can open it outside Jupyter.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio


# This cell is self-contained: it only reads `xy`, `hdb_labels`, `n_clusters`, `noise_frac`,
# and `N` (from Step D, cell 12) plus `_df_manifest` (from cell 2). It does NOT depend on the
# cluster CSV save cell — order of execution is irrelevant as long as Step D has run.

HOVER_COLUMNS = [
    "Image Index", "Image", "Source", "Class", "Subclass", "Specimen Name",
    "Illumination Modality", "logitb",
]
HOVER_TEXT_MAX_CHARS = 80         # truncate long string fields per-cell so the tooltip stays small
PLOT_MARKER_SIZE = 4
PLOT_NOISE_COLOR = "rgba(160,160,160,0.35)"
PLOTLY_RENDERER = None            # None → use the default; set e.g. "notebook_connected" or "browser" if needed

# Build a hover dataframe aligned to the plot rows (here = all N rows in df_manifest order).
_hover_df = _df_manifest.iloc[:N].copy()
for col in HOVER_COLUMNS:
    if col in _hover_df.columns:
        _hover_df[col] = (
            _hover_df[col]
            .astype(str)
            .where(_hover_df[col].notna(), "")
            .str.slice(0, HOVER_TEXT_MAX_CHARS)
        )

def _hover_lines(row_ix: int, cluster_id: int) -> str:
    parts = [f"<b>row</b>: {row_ix}", f"<b>hdbscan_cluster_B</b>: {cluster_id}"]
    for col in HOVER_COLUMNS:
        if col in _hover_df.columns:
            val = _hover_df.iloc[row_ix][col]
            if val == "" or val == "nan":
                continue
            parts.append(f"<b>{col}</b>: {val}")
    return "<br>".join(parts)

# Pre-render hover strings once (5K-ish points → cheap and avoids Plotly's customdata overhead).
_hover_text = [_hover_lines(i, int(hdb_labels[i])) for i in range(N)]

# Color map: -1 → grey; cluster ids → repeating qualitative palette.
_palette = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
_unique_clusters = sorted(set(int(c) for c in hdb_labels))
_color_for = {}
_seen_real = 0
for cid in _unique_clusters:
    if cid == -1:
        _color_for[cid] = PLOT_NOISE_COLOR
    else:
        _color_for[cid] = _palette[_seen_real % len(_palette)]
        _seen_real += 1

fig = go.Figure()

# Plot noise first so it sits underneath colored clusters.
if -1 in _color_for:
    mask = (hdb_labels == -1)
    if mask.any():
        ix_noise = np.where(mask)[0]
        fig.add_trace(go.Scattergl(
            x=xy[ix_noise, 0], y=xy[ix_noise, 1],
            mode="markers",
            marker=dict(size=PLOT_MARKER_SIZE, color=_color_for[-1], line=dict(width=0)),
            name=f"noise (n={len(ix_noise)})",
            text=[_hover_text[i] for i in ix_noise],
            hovertemplate="%{text}<extra></extra>",
            showlegend=True,
        ))

# Then each real cluster as its own trace (so the legend lets you isolate clusters).
for cid in _unique_clusters:
    if cid == -1:
        continue
    mask = (hdb_labels == cid)
    if not mask.any():
        continue
    ix = np.where(mask)[0]
    fig.add_trace(go.Scattergl(
        x=xy[ix, 0], y=xy[ix, 1],
        mode="markers",
        marker=dict(size=PLOT_MARKER_SIZE, color=_color_for[cid], line=dict(width=0)),
        name=f"cluster {cid} (n={len(ix)})",
        text=[_hover_text[i] for i in ix],
        hovertemplate="%{text}<extra></extra>",
        showlegend=True,
    ))

fig.update_layout(
    title=(
        f"B v3 teacher — UMAP (2D) colored by HDBSCAN cluster<br>"
        f"<sub>{n_clusters} clusters | noise = {noise_frac*100:.1f}% | N = {N} | "
        f"LongCLIP-B 16+1 view, fused = {Z.shape[1]}-D</sub>"
    ),
    xaxis_title="UMAP-1",
    yaxis_title="UMAP-2",
    width=1100,
    height=750,
    hovermode="closest",
    legend=dict(itemsizing="constant", font=dict(size=10)),
    template="plotly_white",
)
fig.update_xaxes(scaleanchor="y", scaleratio=1)

if PLOTLY_RENDERER is not None:
    pio.renderers.default = PLOTLY_RENDERER
fig.show()
# To export the interactive plot to a standalone HTML, run in another cell:
#     fig.write_html("teacher_B_v3_umap.html", include_plotlyjs="cdn", full_html=True)

## Step E.1 / E.2 — Uncolored UMAP views (manifold geometry only)

Two diagnostic scatter plots that strip the cluster coloring so you can see the raw UMAP
manifold:

- **E.1 — fused B v3 teacher** (`Z = 2048-D`): same UMAP as Step D's `xy`, drawn as a single
  uniform-color cloud. Useful for judging how clean the fused-teacher manifold is independently
  of the HDBSCAN partition.
- **E.2 — image-only** (`image_only = 512-D`): a separate UMAP fit on the LongCLIP-B image
  vectors alone (weighted-average of random-pool + global, no text fusion), so you can compare
  what the visual side is doing on its own. Hover still shows row metadata for both.

Both cells are self-contained: E.1 reuses `xy` from Step D; E.2 only needs `image_only` from
Step C.

**Thumbnails (E.2):** the code cell immediately after E.2 draws a matplotlib **thumbnail overlay** on the image-only UMAP (random subsample of rows; same idea as `NEHM_pipeline_test.ipynb` / student notebooks).

In [ ]:
# ===== Step E.1 — Fused B v3 teacher UMAP, no coloring =====
# Self-contained: only reads `xy` (from Step D, cell 12), `_df_manifest` (from cell 2), and `N` (cell 2).
import plotly.graph_objects as go
import plotly.io as pio


HOVER_COLUMNS_E1 = [
    "Image Index", "Image", "Source", "Class", "Subclass", "Specimen Name",
    "Illumination Modality", "logitb",
]
HOVER_TEXT_MAX_CHARS_E1 = 80
PLOT_MARKER_SIZE_E1 = 4
PLOT_COLOR_E1 = "rgba(60,90,180,0.55)"   # single uniform color (no cluster encoding)
PLOTLY_RENDERER_E1 = None                 # e.g. "notebook_connected" or "browser" if needed

_hover_df_e1 = _df_manifest.iloc[:N].copy()
for col in HOVER_COLUMNS_E1:
    if col in _hover_df_e1.columns:
        _hover_df_e1[col] = (
            _hover_df_e1[col].astype(str).where(_hover_df_e1[col].notna(), "")
            .str.slice(0, HOVER_TEXT_MAX_CHARS_E1)
        )

def _hover_lines_e1(row_ix: int) -> str:
    parts = [f"<b>row</b>: {row_ix}"]
    for col in HOVER_COLUMNS_E1:
        if col in _hover_df_e1.columns:
            val = _hover_df_e1.iloc[row_ix][col]
            if val == "" or val == "nan":
                continue
            parts.append(f"<b>{col}</b>: {val}")
    return "<br>".join(parts)

_hover_text_e1 = [_hover_lines_e1(i) for i in range(N)]

fig_e1 = go.Figure(go.Scattergl(
    x=xy[:, 0], y=xy[:, 1],
    mode="markers",
    marker=dict(size=PLOT_MARKER_SIZE_E1, color=PLOT_COLOR_E1, line=dict(width=0)),
    name=f"all rows (n={N})",
    text=_hover_text_e1,
    hovertemplate="%{text}<extra></extra>",
))

fig_e1.update_layout(
    title=(
        f"B v3 teacher — UMAP (2D), uncolored<br>"
        f"<sub>fused Z = {Z.shape[1]}-D (LongCLIP-B 16+1 view) | N = {N}</sub>"
    ),
    xaxis_title="UMAP-1", yaxis_title="UMAP-2",
    width=1100, height=750,
    hovermode="closest",
    showlegend=False,
    template="plotly_white",
)
fig_e1.update_xaxes(scaleanchor="y", scaleratio=1)

if PLOTLY_RENDERER_E1 is not None:
    pio.renderers.default = PLOTLY_RENDERER_E1
fig_e1.show()

In [ ]:
# ===== Step E.2 — Image-only UMAP (512-D LongCLIP-B visual, weighted random+global), no coloring =====
# Self-contained: only reads `image_only` (from Step C, cell 10), `_df_manifest` (cell 2), and `N` (cell 2).
# Fits a fresh 2-D UMAP on the image vectors alone (no text fusion) so you can compare what the
# visual side is doing on its own. Same UMAP hyperparameters as Step D for apples-to-apples shape.
import warnings

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import umap


UMAP_N_NEIGHBORS_E2 = 40
UMAP_MIN_DIST_E2    = 0.00
UMAP_METRIC_E2      = "cosine"
UMAP_RANDOM_STATE_E2 = 42

HOVER_COLUMNS_E2 = [
    "Image Index", "Image", "Source", "Class", "Subclass", "Specimen Name",
    "Illumination Modality", "logitb",
]
HOVER_TEXT_MAX_CHARS_E2 = 80
PLOT_MARKER_SIZE_E2 = 4
PLOT_COLOR_E2 = "rgba(180,90,60,0.55)"   # warm color so it visually distinguishes from E.1
PLOTLY_RENDERER_E2 = None

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r".*n_jobs value .* overridden to 1 by setting random_state.*",
        category=UserWarning,
        module=r"umap\..*",
    )
    xy_img = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_N_NEIGHBORS_E2,
        min_dist=UMAP_MIN_DIST_E2,
        metric=UMAP_METRIC_E2,
        random_state=UMAP_RANDOM_STATE_E2,
    ).fit_transform(image_only.astype(np.float32))

_hover_df_e2 = _df_manifest.iloc[:N].copy()
for col in HOVER_COLUMNS_E2:
    if col in _hover_df_e2.columns:
        _hover_df_e2[col] = (
            _hover_df_e2[col].astype(str).where(_hover_df_e2[col].notna(), "")
            .str.slice(0, HOVER_TEXT_MAX_CHARS_E2)
        )

def _hover_lines_e2(row_ix: int) -> str:
    parts = [f"<b>row</b>: {row_ix}"]
    for col in HOVER_COLUMNS_E2:
        if col in _hover_df_e2.columns:
            val = _hover_df_e2.iloc[row_ix][col]
            if val == "" or val == "nan":
                continue
            parts.append(f"<b>{col}</b>: {val}")
    return "<br>".join(parts)

_hover_text_e2 = [_hover_lines_e2(i) for i in range(N)]

fig_e2 = go.Figure(go.Scattergl(
    x=xy_img[:, 0], y=xy_img[:, 1],
    mode="markers",
    marker=dict(size=PLOT_MARKER_SIZE_E2, color=PLOT_COLOR_E2, line=dict(width=0)),
    name=f"all rows (n={N})",
    text=_hover_text_e2,
    hovertemplate="%{text}<extra></extra>",
))

fig_e2.update_layout(
    title=(
        f"B v3 image-only — UMAP (2D), uncolored<br>"
        f"<sub>LongCLIP-B image vectors = {image_only.shape[1]}-D (weighted "
        f"{IMAGE_WEIGHT_RANDOM:.2f}·random + {IMAGE_WEIGHT_GLOBAL:.2f}·global) | N = {N}</sub>"
    ),
    xaxis_title="UMAP-1", yaxis_title="UMAP-2",
    width=1100, height=750,
    hovermode="closest",
    showlegend=False,
    template="plotly_white",
)
fig_e2.update_xaxes(scaleanchor="y", scaleratio=1)

if PLOTLY_RENDERER_E2 is not None:
    pio.renderers.default = PLOTLY_RENDERER_E2
fig_e2.show()

In [ ]:
# ===== Step E.1 — Fused/concatenated UMAP: thumbnail overlay (matplotlib, publication style) =====
# Run the Step D / E.1 cells first (`xy`). Grey field, no axes (UMAP is dimensionless).
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from PIL import Image

if "xy" not in globals():
    raise RuntimeError("Run Step D/E.1 first so `xy` (fused/concatenated UMAP) is defined.")

xy_plot = np.asarray(xy, dtype=np.float64)
if xy_plot.shape[0] != N:
    raise ValueError(f"xy rows {xy_plot.shape[0]} != manifest N={N}")

MAX_THUMBS = 5140
ZOOM = 0.20
# Neutral grey for print; dots slightly darker so the swarm reads on top.
BG_GREY = "#b0b0b0"
POINT_GREY = "#8e8e8e"

rng = np.random.default_rng(42)
perm = rng.permutation(N)
idx = perm[:MAX_THUMBS]
xs = xy_plot[idx, 0]
ys = xy_plot[idx, 1]

fig, ax = plt.subplots(figsize=(11, 11))
fig.patch.set_facecolor(BG_GREY)
ax.set_facecolor(BG_GREY)

dx = xy_plot[:, 0].max() - xy_plot[:, 0].min() + 1e-9
dy = xy_plot[:, 1].max() - xy_plot[:, 1].min() + 1e-9
pad_x, pad_y = 0.05 * dx, 0.05 * dy
ax.set_xlim(xy_plot[:, 0].min() - pad_x, xy_plot[:, 0].max() + pad_x)
ax.set_ylim(xy_plot[:, 1].min() - pad_y, xy_plot[:, 1].max() + pad_y)

ax.scatter(xy_plot[:, 0], xy_plot[:, 1], s=2, alpha=0.35, color=POINT_GREY, linewidths=0)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(axis="both", which="both", length=0, width=0, labelbottom=False, labelleft=False)
for _sp in ax.spines.values():
    _sp.set_visible(False)


def _thumb_rgb(p: Path) -> Image.Image:
    im = Image.open(p).convert("RGB")
    im.thumbnail((96, 96))
    return im


placed = 0
missing = 0
for i, x0, y0 in zip(idx, xs, ys):
    img_name = str(filenames[i]).strip() if i < len(filenames) else ""
    if not img_name:
        missing += 1
        continue
    p = IMAGES_DIR / img_name
    if not p.is_file():
        missing += 1
        continue
    try:
        ab = AnnotationBbox(
            OffsetImage(_thumb_rgb(p), zoom=ZOOM, alpha=1.0),
            (x0, y0),
            frameon=False,
        )
        ax.add_artist(ab)
        placed += 1
    except Exception:
        missing += 1

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()
print(f"Placed thumbnails: {placed} | missing/failed: {missing}")


In [ ]:
# ===== Step E.2 — Image-only UMAP: thumbnail overlay (matplotlib, publication style) =====
# Run the Step E.2 Plotly cell above first (`xy_img`). Grey field, no axes (UMAP is dimensionless).
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from PIL import Image

if "xy_img" not in globals():
    raise RuntimeError("Run the Step E.2 cell first so `xy_img` (image-only UMAP) is defined.")

xy_plot = np.asarray(xy_img, dtype=np.float64)
if xy_plot.shape[0] != N:
    raise ValueError(f"xy_img rows {xy_plot.shape[0]} != manifest N={N}")

MAX_THUMBS = 5140
ZOOM = 0.20
# Neutral grey for print; dots slightly darker so the swarm reads on top.
BG_GREY = "#b0b0b0"
POINT_GREY = "#8e8e8e"

rng = np.random.default_rng(42)
perm = rng.permutation(N)
idx = perm[:MAX_THUMBS]
xs = xy_plot[idx, 0]
ys = xy_plot[idx, 1]

fig, ax = plt.subplots(figsize=(11, 11))
fig.patch.set_facecolor(BG_GREY)
ax.set_facecolor(BG_GREY)

dx = xy_plot[:, 0].max() - xy_plot[:, 0].min() + 1e-9
dy = xy_plot[:, 1].max() - xy_plot[:, 1].min() + 1e-9
pad_x, pad_y = 0.05 * dx, 0.05 * dy
ax.set_xlim(xy_plot[:, 0].min() - pad_x, xy_plot[:, 0].max() + pad_x)
ax.set_ylim(xy_plot[:, 1].min() - pad_y, xy_plot[:, 1].max() + pad_y)

ax.scatter(xy_plot[:, 0], xy_plot[:, 1], s=2, alpha=0.35, color=POINT_GREY, linewidths=0)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(axis="both", which="both", length=0, width=0, labelbottom=False, labelleft=False)
for _sp in ax.spines.values():
    _sp.set_visible(False)

def _thumb_rgb(p: Path) -> Image.Image:
    im = Image.open(p).convert("RGB")
    im.thumbnail((96, 96))
    return im


placed = 0
missing = 0
for i, x0, y0 in zip(idx, xs, ys):
    img_name = str(filenames[i]).strip() if i < len(filenames) else ""
    if not img_name:
        missing += 1
        continue
    p = IMAGES_DIR / img_name
    if not p.is_file():
        missing += 1
        continue
    try:
        ab = AnnotationBbox(
            OffsetImage(_thumb_rgb(p), zoom=ZOOM, alpha=1.0),
            (x0, y0),
            frameon=False,
        )
        ax.add_artist(ab)
        placed += 1
    except Exception:
        missing += 1

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()
print(f"Placed thumbnails: {placed} | missing/failed: {missing}")


In [ ]:
HDBSCAN_CSV_OUT_B = PROJECT_ROOT / "NEHM_RESULTS" / "pipeline_notebook" / "hdbscan_clusters_B.csv"
HDBSCAN_CSV_OUT_B.parent.mkdir(parents=True, exist_ok=True)

save_df = _df_manifest.iloc[:N].copy()
save_df["hdbscan_cluster_B"] = hdb_labels.astype(int)
save_df["umap_x_B"] = xy[:, 0].astype(np.float32)
save_df["umap_y_B"] = xy[:, 1].astype(np.float32)
for _i in range(5):
    save_df[f"umap5_{_i+1}_B"] = z5[:, _i].astype(np.float32)
save_df.to_csv(HDBSCAN_CSV_OUT_B, index=False)
print(f"Saved B HDBSCAN CSV: {HDBSCAN_CSV_OUT_B}")
